In [18]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

def gini_weighted_rf_feature_selection(X_encoded, y, wp=1.0, wn=1.0, F_th=0.030):
    """
    Perform feature selection using Gini impurity based weighted Random Forest.
    
    Parameters:
    - X_encoded: pandas DataFrame or numpy array of encoded features
    - y: target labels
    - wp: weight for positive class
    - wn: weight for negative class
    - F_th: threshold for feature importance; if None, mean importance will be used
    
    Returns:
    - X_selected: pandas DataFrame of selected features
    - selected_features: list of selected feature names
    """
    # Step 1: Load the encoded feature vectors (already in X_encoded)
    # Step 2: Create a blank dictionary D for saving scores (we’ll use a DataFrame for simplicity)

    # Step 3: Instantiate RandomForestClassifier
    model = RandomForestClassifier(
        criterion='gini',
        n_estimators=500,
        class_weight={1: wp, 0: wn},
        random_state=42,
        n_jobs=-1
    )

    # Step 5: Fit model
    model.fit(X_encoded, y)

    # Step 6: Generate feature importance scores
    importances = model.feature_importances_

    # Step 8: Select features above threshold
    selected_indices = np.where(importances >= F_th)[0]
    selected_features = [X_encoded.columns[i] for i in selected_indices]
    
    # Step 9: Return selected features
    X_selected = X_encoded[selected_features]

    return X_selected, selected_features


In [19]:
import pandas as pd
import os
from collections import Counter

input_folder = r'D:\Dataset_Project\dataset_fyp\Network_dataset_preprocess'

# Parameters
chunksize = 100000
n_features_to_select = 20

# Feature selection tracking
feature_counter = Counter()
column_names = None
chunk_count = 0

# Step 1: Loop through CSVs and process in chunks
for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder, filename)

        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            if 'type' in chunk.columns:
                chunk = chunk.drop(columns=['type'])

            if 'label' not in chunk.columns:
                continue  # Skip malformed chunk

            y = chunk['label']
            X = chunk.drop(columns=['label'])

            if column_names is None:
                column_names = X.columns.tolist()

            X = X[column_names]  # Enforce column consistency

            # Feature selection with your method
            _, selected = gini_weighted_rf_feature_selection(X, y)

            # Count selections
            for feat in selected:
                feature_counter[feat] += 1

            chunk_count += 1

# Step 2: Rank features by how often they were selected
selected_features_sorted = feature_counter.most_common(n_features_to_select)
top_features = [feat for feat, _ in selected_features_sorted]

# Step 3: Show feature names and selection counts
print(f"Top {n_features_to_select} features across {chunk_count} chunks (with selection counts):")
for feat, count in selected_features_sorted:
    print(f"{feat}: {count} times")



Top 20 features across 224 chunks (with selection counts):
src_ip_bytes: 221 times
dst_ip_bytes: 220 times
src_pkts: 192 times
src_bytes: 187 times
conn_state: 178 times
proto: 169 times
duration: 166 times
dst_bytes: 166 times
dst_pkts: 164 times
dns_qtype: 134 times
dns_query: 105 times
dns_qclass: 87 times
dns_RD: 57 times
dns_rejected: 47 times
dns_AA: 30 times
service: 26 times
dns_rcode: 23 times
ts: 18 times
dns_RA: 6 times
